# Beginner Tutorial 6: Understanding What Happened (Telemetry)

## What You Will Learn

- What telemetry and observability mean
- Metrics vs. logs vs. traces
- How to read JSONL telemetry files
- How to generate reports
- Using telemetry to identify bottlenecks

## Prerequisites

- Completed notebook 01 (Getting Started)
- `pip install scalable`

In [ ]:
import os
import json
import tempfile
import time

project_dir = tempfile.mkdtemp(prefix="scalable-beginner-06-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

## 💡 Key Concept: What is Telemetry?

**Telemetry** = automated collection of data about what your program did.
From Greek: *tele* (remote) + *metron* (measurement).

Like a flight recorder (black box) for your workflow:
- When did tasks start/finish?
- How much memory/CPU was used?
- Which tasks failed and why?

## 💡 Key Concept: Structured Logging

**Structured logging** = recording events as machine-parseable data (JSON)
instead of free-form text.

Unstructured: `2026-05-20 INFO Task sim(42) done in 4.2s`

Structured:
```json
{"event": "task_completed", "task": "sim", "duration_s": 4.2}
```

Structured logs can be filtered, aggregated, and queried programmatically.

## 💡 Key Concept: JSONL (JSON Lines)

**JSONL** = one JSON object per line. Perfect for event streams:
- Appendable (just add a new line)
- Streamable (process one line at a time)
- Each line is independently parseable

In [ ]:
# First, let's run a workflow to generate telemetry
manifest_content = """\
version: 1
project:
  name: telemetry-demo

targets:
  local:
    provider: local
    max_workers: 2
    threads_per_worker: 1
    processes: false
    containers: none

components:
  analysis:
    cpus: 1
    memory: 1G

tasks:
  run_analysis:
    component: analysis
"""

with open("scalable.yaml", "w") as f:
    f.write(manifest_content)
print("Manifest ready. Running workflow to generate telemetry...")

In [ ]:
import random
from scalable import ScalableSession

def variable_work(task_id: int) -> dict:
    """Task with variable duration (to make telemetry interesting)."""
    duration = 0.2 + random.random() * 0.8  # 0.2 to 1.0 seconds
    time.sleep(duration)
    return {"task_id": task_id, "duration": duration}

# Run workflow
session = ScalableSession.from_yaml("./scalable.yaml", target="local")
plan = session.plan()
client = session.start(plan)
futures = [client.submit(variable_work, i, tag="analysis") for i in range(10)]
results = client.gather(futures)
session.close()

print(f"Completed {len(results)} tasks")
print("Telemetry has been recorded automatically!")

In [ ]:
# Explore the telemetry directory structure
scalable_dir = ".scalable/runs"
if os.path.exists(scalable_dir):
    runs = os.listdir(scalable_dir)
    print(f"Found {len(runs)} run(s) in .scalable/runs/")
    if runs:
        run_dir = os.path.join(scalable_dir, sorted(runs)[-1])
        print(f"\nLatest run: {os.path.basename(run_dir)}")
        print("\nFiles:")
        for f in sorted(os.listdir(run_dir)):
            size = os.path.getsize(os.path.join(run_dir, f))
            print(f"  {f} ({size} bytes)")
else:
    print("No telemetry directory found (session may not have written telemetry in this mode)")
    print("\n💡 In production usage, telemetry is always written.")
    print("Let's simulate what telemetry data looks like...")

In [ ]:
# Simulate telemetry data to show the format
# (In production, this is generated automatically)
import datetime

simulated_events = []
base_time = datetime.datetime.now(datetime.timezone.utc)

for i, r in enumerate(results):
    start_time = base_time + datetime.timedelta(seconds=i * 0.1)
    end_time = start_time + datetime.timedelta(seconds=r['duration'])
    
    simulated_events.append({
        "event": "task_completed",
        "task": "run_analysis",
        "task_id": r['task_id'],
        "timestamp": end_time.isoformat(),
        "duration_s": round(r['duration'], 3),
        "worker": f"worker-{i % 2}"
    })

print("Example telemetry events (JSONL format):")
print("─" * 60)
for event in simulated_events[:5]:
    print(json.dumps(event))
print("...")
print(f"\n💡 Each line is a complete JSON object — that's JSONL!")

In [ ]:
# Analyze the telemetry data
durations = [e['duration_s'] for e in simulated_events]

print("Task Duration Analysis:")
print(f"  Total tasks: {len(durations)}")
print(f"  Average duration: {sum(durations)/len(durations):.3f}s")
print(f"  Fastest task: {min(durations):.3f}s")
print(f"  Slowest task: {max(durations):.3f}s")
print(f"\nWorker Distribution:")
for w in ['worker-0', 'worker-1']:
    count = sum(1 for e in simulated_events if e['worker'] == w)
    print(f"  {w}: {count} tasks")

print(f"\n💡 Use this data to optimize: are workers balanced? Any outlier tasks?")

## 📖 Vocabulary Summary

| Term | Definition |
|------|------------|
| Telemetry | Automated collection of system behavior data |
| Observability | Ability to understand internal state from outputs |
| Metrics | Numerical measurements over time |
| Logs | Discrete events with context |
| Traces | Journey of a request through the system |
| Structured Logging | Machine-parseable event recording (JSON) |
| JSONL | JSON Lines — one JSON object per line |
| Event | Discrete occurrence with timestamp and payload |
| Utilization | Percentage of resources actually being used |

In [ ]:
# Cleanup
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print(f"Cleaned up {project_dir}")